# Study 938 — Open or Close — the teardown

The sliver algebra, per-tape HAC *t* on two rules, block-bootstrap CIs, the era cut, the entry-minus-exit intraday decomposition, both PROXY sweeps, and the live synthetic control. Every real number is frozen from `docs/results.md` (Fingerprint `f36d90ae4fdc`, as-of 2026-06-30); the synthetic cells are labelled.

In [1]:
R = {'start': '2007-05-30', 'end': '2026-06-30', 'n_days': 4802, 'fp': 'f36d90ae4fdc', 'cost_bps': 2.0, 'split': '2017-01-01', 'm_tapes': {'SPY': (31, -4.8, -0.16, -2.9, 0.45, 0.587, 0.589), 'IWM': (37, 50.7, 1.04, 25.1, 0.59, 0.363, 0.331), 'EEM': (37, 66.6, 2.4, 32.9, 0.7, 0.313, 0.27), 'EFA': (33, 35.4, 1.69, 19.6, 0.61, 0.299, 0.272)}, 'm_gap': 37.0, 'm_t': 1.53, 'm_ci': (-12.0, 85.3), 'm_ci_neg': 6.8, 'm_sh_open': 0.444, 'm_sh_close': 0.413, 'm_cagr_open': 4.9, 'm_cagr_close': 4.51, 'm_cagr_open_tot': 6.21, 'm_cagr_close_tot': 5.82, 'm_era_early': (57.4, 1.96), 'm_era_late': (17.9, 0.47), 'w_tapes': {'SPY': (88, -35.8, -0.81, -7.6, 0.5, 0.588, 0.615), 'IWM': (114, -19.0, -0.29, -3.1, 0.46, 0.382, 0.392), 'EEM': (102, -26.2, -0.54, -4.8, 0.45, 0.333, 0.349), 'EFA': (96, -23.4, -0.66, -4.6, 0.45, 0.412, 0.432)}, 'w_gap': -26.1, 'w_t': -0.8, 'w_ci': (-90.9, 35.7), 'w_ci_neg': 80.7, 'w_sh_open': 0.494, 'w_sh_close': 0.515, 'w_cagr_open': 5.27, 'w_cagr_close': 5.54, 'w_cagr_open_tot': 6.64, 'w_cagr_close_tot': 6.92, 'w_era_early': (27.2, 0.58), 'w_era_late': (-78.0, -1.73), 'n_trades': 538, 'per_trade_bps': 1.39, 'per_trade_t': 0.33, 'wins': 267, 'win_rate': 49.6, 'wilson': (45.4, 53.8), 'n_dates': 361, 'per_trade_t_clu': 0.23, 'clu_mean_ci': (-10.8, 13.0), 'clu_win': (44.1, 55.1), 'disp_mean': 5.4, 'disp_sd': 39.5, 'disp_lo': -35.8, 'disp_hi': 66.6, 'disp_fires': 1, 'pen': {0.0: ((37.0, 1.53), (-26.1, -0.8)), 1.0: ((35.1, 1.46), (-31.5, -0.97)), 2.0: ((33.2, 1.38), (-36.8, -1.13)), 5.0: ((27.5, 1.15), (-52.9, -1.62))}, 'cash_sweep': {0.0: -4.87, 0.5: -4.84, 0.73: -4.85, 1.0: -4.89}, 'mech_m': {'SPY': (31.9, 39.8, -7.9, -0.22), 'IWM': (0.9, -50.4, 51.2, 1.06), 'EEM': (48.0, -17.1, 65.1, 2.51), 'EFA': (25.2, -13.5, 38.7, 1.68)}, 'mech_w_spreads': {'SPY': (-15.4, -0.82), 'IWM': (-6.4, -0.29), 'EEM': (-9.9, -0.57), 'EFA': (-9.4, -0.69)}, 'syn_planted': (134.6, 28.2, 3.84, 8), 'syn_null': (13.8, 38.4, 0.55, 1)}

## Setup — exactly one execution lag, two venues

Let `target[t]` be the rule's weight formed from data through the close of day `t`. Both arms fill on `t+1`:

* **open arm** — overnight leg of `t+1` at `target[t-1]`, intraday leg at `target[t]`;
* **close arm** — the whole of `t+1` at `target[t-1]`, `target[t]` from that close.

Hence on any day `u`,

```
r_open[u] - r_close[u] = (w_id[u] - w_on[u]) * (r_intraday[u] - r_intraday_cash[u])
                         + O((daily return)^2)
```

which is exactly zero whenever `w_id == w_on` — i.e. on every non-trade day. The study is therefore a test on ~540 observations, not ~4,800, and its power is capped by the rules' turnover.

> 💡 **In plain words** — the two versions of the portfolio own the same thing except on the days the rule flips, and on those days they differ by one trading session.

## Headline — 10-month rule (month-end rebalance), 2 bps one-way

In [2]:
print(f"{'tape':5s} {'trades':>7s} {'gap bps/yr':>11s} {'HAC t':>7s} "
      f"{'per-trade':>10s} {'open wins':>10s}  exSharpe open/close")
for tk, v in R['m_tapes'].items():
    print(f"{tk:5s} {v[0]:7d} {v[1]:+11.1f} {v[2]:+7.2f} {v[3]:+10.1f} "
          f"{v[4]:10.0%}  {v[5]:+.3f} / {v[6]:+.3f}")
print(f"\npooled (EW 4 tapes): {R['m_gap']:+.1f} bps/yr  HAC t {R['m_t']:+.2f}  "
      f"95% block-bootstrap CI [{R['m_ci'][0]:+.1f}, {R['m_ci'][1]:+.1f}] "
      f"(share<0 {R['m_ci_neg']:.1f}%)")
print(f"pooled exSharpe: open {R['m_sh_open']:+.3f} vs close {R['m_sh_close']:+.3f} "
      f"(d {R['m_sh_open']-R['m_sh_close']:+.3f})")
print(f"excess-of-cash CAGR {R['m_cagr_open']:.2f}% vs {R['m_cagr_close']:.2f}%  |  "
      f"raw, cash-inclusive {R['m_cagr_open_tot']:.2f}% vs {R['m_cagr_close_tot']:.2f}%")

tape   trades  gap bps/yr   HAC t  per-trade  open wins  exSharpe open/close
SPY        31        -4.8   -0.16       -2.9        45%  +0.587 / +0.589
IWM        37       +50.7   +1.04      +25.1        59%  +0.363 / +0.331
EEM        37       +66.6   +2.40      +32.9        70%  +0.313 / +0.270
EFA        33       +35.4   +1.69      +19.6        61%  +0.299 / +0.272

pooled (EW 4 tapes): +37.0 bps/yr  HAC t +1.53  95% block-bootstrap CI [-12.0, +85.3] (share<0 6.8%)
pooled exSharpe: open +0.444 vs close +0.413 (d +0.031)
excess-of-cash CAGR 4.90% vs 4.51%  |  raw, cash-inclusive 6.21% vs 5.82%


Gross equals net here: both venues pay the same 2 bps one-way × NAV on the same trade days, and there is no short leg, so no borrow. Only EEM clears |*t*| = 2 — on 37 trades. Both growth rates are printed and kept apart: the Sharpes are built on the **excess-of-cash** series, and the raw figure is ~1.3 pp/yr higher purely because the book sits in T-bills for a large share of the sample.

## A faster cousin, three times the fills — 20-week (week-end rebalance)

Twenty weeks is ≈ 100 sessions of lookback against ≈ 210 for ten months, so this arm is shorter-horizon as well as higher-frequency. It is the power check, not a clean replication.

In [3]:
print(f"{'tape':5s} {'trades':>7s} {'gap bps/yr':>11s} {'HAC t':>7s} "
      f"{'per-trade':>10s} {'open wins':>10s}  exSharpe open/close")
for tk, v in R['w_tapes'].items():
    print(f"{tk:5s} {v[0]:7d} {v[1]:+11.1f} {v[2]:+7.2f} {v[3]:+10.1f} "
          f"{v[4]:10.0%}  {v[5]:+.3f} / {v[6]:+.3f}")
print(f"\npooled: {R['w_gap']:+.1f} bps/yr  HAC t {R['w_t']:+.2f}  "
      f"95% CI [{R['w_ci'][0]:+.1f}, {R['w_ci'][1]:+.1f}] (share<0 {R['w_ci_neg']:.1f}%)")
print(f"pooled exSharpe: open {R['w_sh_open']:+.3f} vs close {R['w_sh_close']:+.3f} "
      f"(d {R['w_sh_open']-R['w_sh_close']:+.3f})")
print(f"excess-of-cash CAGR {R['w_cagr_open']:.2f}% vs {R['w_cagr_close']:.2f}%  |  "
      f"raw, cash-inclusive {R['w_cagr_open_tot']:.2f}% vs {R['w_cagr_close_tot']:.2f}%")
print('\nall four tapes reverse sign relative to the monthly rule.')

tape   trades  gap bps/yr   HAC t  per-trade  open wins  exSharpe open/close
SPY        88       -35.8   -0.81       -7.6        50%  +0.588 / +0.615
IWM       114       -19.0   -0.29       -3.1        46%  +0.382 / +0.392
EEM       102       -26.2   -0.54       -4.8        45%  +0.333 / +0.349
EFA        96       -23.4   -0.66       -4.6        45%  +0.412 / +0.432

pooled: -26.1 bps/yr  HAC t -0.80  95% CI [-90.9, +35.7] (share<0 80.7%)
pooled exSharpe: open +0.494 vs close +0.515 (d -0.021)
excess-of-cash CAGR 5.27% vs 5.54%  |  raw, cash-inclusive 6.64% vs 6.92%

all four tapes reverse sign relative to the monthly rule.


## Pooled over every fill, and the era cut

The pooled sample is **not** 538 independent draws: SPY / IWM / EEM / EFA are highly correlated and their moving-average signals flip on the same days, so the fills sit on 361 distinct dates, and the two rules share one window. The naive *t* and the Wilson interval assume that away; clustering on the trade date is the minimum correction, and it is reported next to them.

In [4]:
print(f"{R['n_trades']} fills pooled: mean {R['per_trade_bps']:+.2f} bps "
      f"(naive t = {R['per_trade_t']:+.2f}), open wins {R['wins']}/{R['n_trades']} = "
      f"{R['win_rate']:.1f}%, 95% Wilson [{R['wilson'][0]:.1f}%, {R['wilson'][1]:.1f}%]")
print(f"clustered on trade date ({R['n_dates']} dates): t = {R['per_trade_t_clu']:+.2f}, "
      f"mean 95% CI [{R['clu_mean_ci'][0]:+.1f}, {R['clu_mean_ci'][1]:+.1f}] bps, "
      f"win rate 95% CI [{R['clu_win'][0]:.1f}%, {R['clu_win'][1]:.1f}%]")
print(f"\nera cut at {R['split']} (pooled book):")
print(f"  10-month : early {R['m_era_early'][0]:+7.1f} (t {R['m_era_early'][1]:+.2f})   "
      f"late {R['m_era_late'][0]:+7.1f} (t {R['m_era_late'][1]:+.2f})")
print(f"  20-week  : early {R['w_era_early'][0]:+7.1f} (t {R['w_era_early'][1]:+.2f})   "
      f"late {R['w_era_late'][0]:+7.1f} (t {R['w_era_late'][1]:+.2f})")

538 fills pooled: mean +1.39 bps (naive t = +0.33), open wins 267/538 = 49.6%, 95% Wilson [45.4%, 53.8%]
clustered on trade date (361 dates): t = +0.23, mean 95% CI [-10.8, +13.0] bps, win rate 95% CI [44.1%, 55.1%]

era cut at 2017-01-01 (pooled book):
  10-month : early   +57.4 (t +1.96)   late   +17.9 (t +0.47)
  20-week  : early   +27.2 (t +0.58)   late   -78.0 (t -1.73)


> 💡 **In plain words** — no half of the sample agrees with the other, and the two rules disagree with each other in the recent era.

## Mechanism — the entry-minus-exit intraday spread

Entries carry `Δw = +1`, exits `Δw = −1`, and they alternate. A *constant* intraday drift therefore cancels over a full cycle; a venue edge requires the **conditional** means to differ. Mean intraday (price-only) return on trade days:

In [5]:
print('10-month rule:')
print(f"  {'tape':5s} {'entry bps':>10s} {'exit bps':>10s} {'spread':>9s} {'Welch t':>8s}")
for tk, v in R['mech_m'].items():
    print(f"  {tk:5s} {v[0]:+10.1f} {v[1]:+10.1f} {v[2]:+9.1f} {v[3]:+8.2f}")
print('\n20-week rule (entry - exit spread only):')
for tk, v in R['mech_w_spreads'].items():
    print(f"  {tk:5s} {v[0]:+9.1f} bps  (Welch t {v[1]:+.2f})")
print('\nall four spreads flip negative on the higher-powered rule.')

10-month rule:
  tape   entry bps   exit bps    spread  Welch t
  SPY        +31.9      +39.8      -7.9    -0.22
  IWM         +0.9      -50.4     +51.2    +1.06
  EEM        +48.0      -17.1     +65.1    +2.51
  EFA        +25.2      -13.5     +38.7    +1.68

20-week rule (entry - exit spread only):
  SPY       -15.4 bps  (Welch t -0.82)
  IWM        -6.4 bps  (Welch t -0.29)
  EEM        -9.9 bps  (Welch t -0.57)
  EFA        -9.4 bps  (Welch t -0.69)

all four spreads flip negative on the higher-powered rule.


The one monthly spread that clears |*t*| = 2 (EEM, +65.1 bps, Welch *t* = +2.51, on 19 entries against 18 exits) does not survive the switch to weekly rebalancing, where the same tape's spread is **−9.9 bps** (*t* = −0.57). That is the signature of a small-sample artefact rather than a mechanism.

## Non-tape inputs — both PROXIES, both swept

**(1) Opening-auction penalty.** The headline charges both venues the same 2 bps, which flatters the open (the opening auction is the wider, thinner book). We sweep an *extra* one-way charge on the open arm only. **(2) Cash split.** BIL's printed open is sub-penny noise, so the daily T-bill accrual is split night/day by an assumed fraction (default 0.73).

In [6]:
print('opening-auction penalty (PROXY) — extra one-way bps on the open arm only:')
print(f"  {'extra':>6s} {'10-month gap (t)':>22s} {'20-week gap (t)':>22s}")
for x, (mm, ww) in sorted(R['pen'].items()):
    print(f"  {x:6.1f} {mm[0]:+13.1f} ({mm[1]:+.2f}) {ww[0]:+13.1f} ({ww[1]:+.2f})")
print('\ncash-split PROXY (SPY, 10-month) — overnight share of the T-bill accrual:')
for f, g in sorted(R['cash_sweep'].items()):
    print(f"  f={f:.2f} -> gap {g:+.2f} bps/yr")
print('  -> inert, as a ~1 bp/day accrual must be.')

opening-auction penalty (PROXY) — extra one-way bps on the open arm only:
   extra       10-month gap (t)        20-week gap (t)
     0.0         +37.0 (+1.53)         -26.1 (-0.80)
     1.0         +35.1 (+1.46)         -31.5 (-0.97)
     2.0         +33.2 (+1.38)         -36.8 (-1.13)
     5.0         +27.5 (+1.15)         -52.9 (-1.62)

cash-split PROXY (SPY, 10-month) — overnight share of the T-bill accrual:
  f=0.00 -> gap -4.87 bps/yr
  f=0.50 -> gap -4.84 bps/yr
  f=0.73 -> gap -4.85 bps/yr
  f=1.00 -> gap -4.89 bps/yr
  -> inert, as a ~1 bp/day accrual must be.


Both rules move toward the **close** as the opening spread is charged more honestly. That is the only monotone, sign-stable statement in the study — and it is a cost argument, not an alpha one.

## Dispersion — the cost of an arbitrary choice

In [7]:
print(f"across the 8 rule x tape cells:")
print(f"  mean {R['disp_mean']:+.1f} bps/yr, sd {R['disp_sd']:.1f}, "
      f"range [{R['disp_lo']:+.1f}, {R['disp_hi']:+.1f}]")
print(f"  |t| >= 2 in {R['disp_fires']}/8 cells (chance would give ~0.4)")
print(f"  -> ~+/-{R['disp_sd']/100:.1f} pp/yr of unforecastable track-record luck")

across the 8 rule x tape cells:
  mean +5.4 bps/yr, sd 39.5, range [-35.8, +66.6]
  |t| >= 2 in 1/8 cells (chance would give ~0.4)
  -> ~+/-0.4 pp/yr of unforecastable track-record luck


## Live synthetic control (offline — NOT the real tape)

The generator fixes the close-to-close path and only re-splits it between the overnight and intraday legs, so the close-executed arm is bit-identical across the knob. `signal_strength=1` plants intraday continuation after strong/weak months (the open fill must win); `signal_strength=0` randomises the split (the venues must tie).

In [8]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from open_close_exec import data, strategy as st
for ss, tag in [(1.0, 'planted continuation'), (0.0, 'null')]:
    rows = [st.synthetic_detect(
                data.synthetic_daily(signal_strength=ss, seed=938 + 11 * s)[0])
            for s in range(8)]
    g = np.array([r['gap_ann_bps'] for r in rows])
    t = np.array([r['gap_t_hac'] for r in rows])
    print(f"SYNTHETIC {tag:22s}: gap {g.mean():+7.1f} bps/yr (sd {g.std(ddof=1):.1f}), "
          f"mean HAC t {t.mean():+.2f}, fires {int((abs(t) >= 2).sum())}/8")
print('\n(frozen reference from docs/results.md: planted %+.1f / t %+.2f / %d of 8;'
      ' null %+.1f / t %+.2f / %d of 8)'
      % (R['syn_planted'][0], R['syn_planted'][2], R['syn_planted'][3],
         R['syn_null'][0], R['syn_null'][2], R['syn_null'][3]))

SYNTHETIC planted continuation  : gap  +134.6 bps/yr (sd 28.2), mean HAC t +3.84, fires 8/8


SYNTHETIC null                  : gap   +13.8 bps/yr (sd 38.4), mean HAC t +0.55, fires 1/8

(frozen reference from docs/results.md: planted +134.6 / t +3.84 / 8 of 8; null +13.8 / t +0.55 / 1 of 8)


## Verdict

- **Signal — None.** Pooled over 538 fills (on 361 distinct dates) the open-minus-close slippage is **+1.39 bps**, *t* = +0.33 naive and **+0.23 clustered on the trade date**; win rate **49.6%** with 95% Wilson [45.4%, 53.8%] — cluster-bootstrap [44.1%, 55.1%]. The 10-month rule says +37.0 bps/yr (*t* = +1.53), the higher-powered 20-week rule says -26.1 (*t* = -0.80); both bootstrap CIs contain zero; the era cut reverses; and the one significant cell's mechanism (EEM's entry-minus-exit intraday spread) flips sign under more data. The synthetic control fires 8/8 on a planted venue edge and 1/8 on the null, so the harness is awake. *Survivorship: none — four continuously listed index ETFs; intraday legs are price-only, overnight legs carry the dividend, labelled throughout.*
- **Tradability — Mirage.** No committable sign, hence no venue rule. The residual is dispersion — realised gaps spanning [-35.8, +66.6] bps/yr across the eight cells, ~±0.4 pp/yr of free luck. The one robust statement is the friction one: charge the opening auction its real spread and both rules prefer the **close**.